In [ ]:
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("HF token: ")

In [ ]:
print(bool(os.environ.get("HF_TOKEN")))   # має бути True

In [ ]:

import torch; print(torch.cuda.is_available())

In [ ]:
!git clone https://github.com/Luzefik/storozha-anti-ambush.git

In [ ]:
!ls



In [ ]:
cd storozha-anti-ambush

In [ ]:
ls

In [ ]:
!git checkout feat/project-scaffold
!git log --oneline -1
!pip install -q -e . --no-deps
import os
import sys
sys.path.insert(0, os.path.abspath("src"))   # pip install -e не видно в уже запущеному ядрі без рестарту


In [ ]:
!pip install ultralytics
!pip install -q kaggle
!kaggle competitions download -c predicting-assessment-scores-hw1-ucu-ai-2026 -p data
!unzip -q -o data/*.zip -d data
!ls data
!pip install -q pandas numpy seaborn matplotlib scikit-learn xgboost
!pip install seaborn


In [ ]:
!python scripts/download_data.py
!python scripts/prepare_data.py

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV,
                                     cross_val_predict, cross_validate)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from ultralytics import YOLO

In [ ]:

import subprocess
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/storozha/runs"
Path(RUNS).mkdir(parents=True, exist_ok=True)


def log_run(name, runs=RUNS):
    df = pd.read_csv(f"{runs}/{name}/results.csv")
    df.columns = df.columns.str.strip()
    best = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
    last10 = df.tail(10)
    row = {
        "run": name,
        "epochs_run": len(df),
        "best_epoch": int(best["epoch"]),
        "best_mAP50": round(best["metrics/mAP50(B)"], 4),
        "best_recall": round(best["metrics/recall(B)"], 4),
        "last10_mAP50": round(last10["metrics/mAP50(B)"].mean(), 4),
        "git_commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                     capture_output=True, text=True).stdout.strip(),
    }
    log = Path(runs) / "summary.csv"
    pd.DataFrame([row]).to_csv(log, mode="a", header=not log.exists(), index=False)
    print(row)
    return row

In [ ]:
from ultralytics import YOLO

RUN = "yolo26n-640"
model = YOLO("yolo26n.pt")
model.train(data="data/processed/yolo/data.yaml",
            imgsz=640, epochs=100, patience=20, batch=-1, seed=42,
            device=0, cache="ram", workers=2,
            project=RUNS, name=RUN)   # -> /content/drive/MyDrive/storozha/runs/<RUN>
RUN = Path(model.trainer.save_dir).name
log_run(RUN)

In [ ]:
from ultralytics import RTDETR

RUN = "rtdetr-l-640"
model = RTDETR("rtdetr-l.pt")
model.train(data="data/processed/yolo/data.yaml",
            imgsz=640, epochs=100, patience=20, batch=8, seed=42,
            device=0, workers=2,
            project=RUNS, name=RUN)
RUN = Path(model.trainer.save_dir).name
log_run(RUN)

In [ ]:
from ultralytics import YOLO, RTDETR

RUN = "yolo26n-1280"
weights = f"{RUNS}/{RUN}/weights/best.pt"
Model = RTDETR if RUN.startswith("rtdetr") else YOLO
model = Model(weights)

# відео з test-спліту; результат (відео з рамками) — лише в приватну папку на Drive
video = "data/raw/Z-FPV-v1/videos/video-24.mp4"
results = model.predict(source=video, imgsz=1280, conf=0.25, device=0,
                        stream=True, save=True,
                        project="/content/drive/MyDrive/storozha/predictions",
                        name=f"{RUN}_video-24")
n_frames = n_det = 0
for r in results:          # зі stream=True обробка йде лише під час перебору
    n_frames += 1
    n_det += len(r.boxes) > 0
print(f"кадрів: {n_frames}, з детекцією: {n_det}")

In [ ]:

from pathlib import Path
import yaml


yolo = Path("data/processed/yolo")
m = pd.read_csv("config/splits/video_split_manifest.csv")
pool = m[m["split"] != "test"]

K = 4
order = pd.concat([pool[pool.is_night], pool[~pool.is_night].sort_values("n_frames", ascending=False)])
folds = [[] for _ in range(K)]
for i, v in enumerate(order["video"]):
    folds[i % K].append(v)

def lines(videos):
    return [f"./images/{v}/{p.name}" for v in videos
            for p in sorted((yolo / "images" / v).glob("frame_*.png"))]

for k, val_videos in enumerate(folds):
    train_videos = [v for v in pool["video"] if v not in val_videos]
    (yolo / f"fold{k}_train.txt").write_text("\n".join(lines(train_videos)) + "\n")
    (yolo / f"fold{k}_val.txt").write_text("\n".join(lines(val_videos)) + "\n")
    (yolo / f"fold{k}.yaml").write_text(yaml.safe_dump(
        {"train": f"fold{k}_train.txt", "val": f"fold{k}_val.txt",
         "names": {0: "waiter_fpv_drone"}}, sort_keys=False))
    print(k, "val:", val_videos)

In [ ]:
import numpy as np
from ultralytics import YOLO

CV_NAME = "cv-yolo11s-1240"                 # назва всього експерименту з крос-валідацією
CV_DIR = f"{RUNS}/{CV_NAME}"
fold_maps = []
for k in range(K):
    model = YOLO("yolo11s.pt")
    model.train(data=f"data/processed/yolo/fold{k}.yaml", imgsz=1240, epochs=50,
                patience=15, batch=-1, seed=42, device=0, cache="ram", workers=2,
                project=CV_DIR, name=f"fold{k}")
    fold_maps.append(model.metrics.box.map50)          # mAP@0.5 на val цього фолду
    log_run(Path(model.trainer.save_dir).name, runs=CV_DIR)   # summary.csv усередині CV_DIR

print(f"mAP50 = {np.mean(fold_maps):.3f} ± {np.std(fold_maps):.3f}", fold_maps)
pd.DataFrame({"fold": range(K), "mAP50": fold_maps}).to_csv(f"{CV_DIR}/cv_results.csv", index=False)

## Оцінка спільним evaluator

Одна функція `evaluate` (`src/zfpv/eval/metrics.py`) для всіх моделей: AP@0.5 / AP@0.75, precision і recall окремо для кожного відео і для день/ніч. Налаштування беруться з `config/default.yaml → eval`, тож вони однакові для всіх моделей.
Результат зберігається на Drive поруч із вагами (`eval_val.json`). У репозиторій переносимо з ноутбука, а не з Colab.
**Test не чіпаємо** до фінальної моделі: тут лише `VAL` (або val-відео фолду).

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath("src"))   # якщо ядро не перезапускали після pip install -e
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import yaml
from ultralytics import RTDETR, YOLO

from zfpv.config import load_config
from zfpv.data.tiling import boxes_in_tile, predict_tiled, tile_origins
from zfpv.eval.metrics import evaluate

cfg = load_config()
EV = cfg["eval"]                                   # однакові налаштування для всіх моделей
YOLO_DIR = Path(cfg["paths"]["yolo_dir"])
manifest = pd.read_csv(cfg["paths"]["split_manifest"])
DAY_NIGHT = {r.video: "night" if r.is_night else "day" for r in manifest.itertuples()}
VAL = manifest.loc[manifest.split == "val", "video"].tolist()


def load_gt(video, every=1):
    """Кадри одного відео і рамки дронів у пікселях [x1, y1, x2, y2]."""
    paths = sorted((YOLO_DIR / "images" / video).glob("frame_*.png"))[::every]
    h, w = cv2.imread(str(paths[0])).shape[:2]
    boxes = []
    for p in paths:
        text = (YOLO_DIR / "labels" / video / f"{p.stem}.txt").read_text()
        rows = np.array(text.split(), dtype=float).reshape(-1, 5)    # cls xc yc w h, від 0 до 1
        xc, yc, bw, bh = rows[:, 1] * w, rows[:, 2] * h, rows[:, 3] * w, rows[:, 4] * h
        boxes.append(np.stack([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], axis=1))
    return paths, boxes


def run_eval(weights, videos, imgsz=640, tile=None, every=1, batch=16):
    """Модель -> передбачення на кадрах -> спільний evaluator. tile=640 вмикає SAHI."""
    model = (RTDETR if "rtdetr" in str(weights) else YOLO)(weights)

    def predict(images):                           # -> список (K, 5): x1, y1, x2, y2, score
        res = model.predict(images, imgsz=imgsz, conf=0.001, max_det=EV["max_detections"],
                            device=0, verbose=False)
        return [np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None]])
                for r in res]

    preds, gts = {}, {}
    for v in videos:
        paths, gts[v] = load_gt(v, every)
        preds[v] = []
        for i in range(0, len(paths), batch):
            frames = [cv2.imread(str(p)) for p in paths[i:i + batch]]   # BGR, як чекає Ultralytics
            preds[v] += [predict_tiled(predict, f, tile) for f in frames] if tile else predict(frames)
    return evaluate(preds, gts, EV["iou_thresholds"], EV["conf_threshold"],
                    EV["max_detections"], groups=DAY_NIGHT)


def show(res):
    """Таблиця: кожне відео, день/ніч і всі разом."""
    t = pd.DataFrame({**res["per_video"], **res["per_group"], "усі": res["overall"]}).T
    return t[["n_gt", "AP@0.5", "AP@0.75", "precision@0.5", "recall@0.5"]].astype(float).round(3)

In [ ]:
RUN, IMGSZ = "yolo26n-1280-3", 1280                # який запуск оцінюємо і з яким imgsz його вчили
res = run_eval(f"{RUNS}/{RUN}/weights/best.pt", VAL, imgsz=IMGSZ)
json.dump(res, open(f"{RUNS}/{RUN}/eval_val.json", "w"), indent=2)
show(res)

In [ ]:
# Перевірка evaluator: AP@0.5 вище має бути близьким до mAP50 самого Ultralytics на тому ж val.
# Якщо різниця велика (> ~0.03), десь помилка в рамках, і цифрам не можна вірити.
YOLO(f"{RUNS}/{RUN}/weights/best.pt").val(data=str(YOLO_DIR / "data.yaml"), imgsz=IMGSZ,
                                          device=0, verbose=False).box.map50

## SAHI: оцінка по тайлах

Кадр ріжеться на тайли 640×640 з перекриттям 20%, модель бачить кожен тайл у рідній роздільності. Рамки з тайлів переносяться назад у кадр, а дублікати склеюються через NMS (`src/zfpv/data/tiling.py`). Цілий кадр теж передбачається, щоб не загубити великих дронів на межі тайлів. Модель та сама, що вище: змінюється лише спосіб передбачення.

In [ ]:
res_sahi = run_eval(f"{RUNS}/{RUN}/weights/best.pt", VAL, imgsz=640, tile=640)
json.dump(res_sahi, open(f"{RUNS}/{RUN}/eval_val_sahi640.json", "w"), indent=2)
pd.concat({"цілий кадр": show(res), "SAHI 640": show(res_sahi)}, axis=1)

## SAHI: навчання на тайлах

Train-кадри нарізаються на тайли 640×640: модель вчиться на дронах у рідньому розмірі. Беремо кожен 3-й кадр (сусідні майже однакові) і лише 20% порожніх тайлів, бо інакше їх була б більшість. Val лишається на цілих кадрах. Тайли лежать у `data/processed`, тож git їх не бачить; **не комітити** (ліцензія датасету).
Під час навчання Ultralytics рахує val на цілих кадрах при `imgsz=640`, а це занижує результат. Чесна цифра — `run_eval(..., tile=640)` наприкінці.

In [ ]:
TILE, EVERY, NEG_KEEP = 640, 3, 0.2
TILES = YOLO_DIR.parent / f"tiles{TILE}"
rng = np.random.default_rng(cfg["seed"])
lines = []
for v in manifest.loc[manifest.split == "train", "video"]:
    (TILES / "images" / v).mkdir(parents=True, exist_ok=True)
    (TILES / "labels" / v).mkdir(parents=True, exist_ok=True)
    paths, boxes = load_gt(v, every=EVERY)
    for p, b in zip(paths, boxes):
        frame = cv2.imread(str(p))
        for x0, y0 in tile_origins(frame.shape[1], frame.shape[0], TILE, overlap=0.2):
            tb = boxes_in_tile(b, x0, y0, TILE, min_visible=0.5)
            if len(tb) == 0 and rng.random() > NEG_KEEP:
                continue
            crop = frame[y0:y0 + TILE, x0:x0 + TILE]
            h, w = crop.shape[:2]
            name = f"{p.stem}_x{x0}_y{y0}"
            cv2.imwrite(str(TILES / "images" / v / f"{name}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
            labels = [f"0 {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}"
                      for x1, y1, x2, y2 in tb]
            (TILES / "labels" / v / f"{name}.txt").write_text("\n".join(labels))
            lines.append(f"./images/{v}/{name}.jpg")
(TILES / "train.txt").write_text("\n".join(lines) + "\n")
(TILES / "data.yaml").write_text(yaml.safe_dump(
    {"train": "train.txt", "val": "../yolo/val.txt", "names": {0: "waiter_fpv_drone"}}))
print(len(lines), "тайлів ->", TILES)

In [ ]:
RUN = "yolo26n-tiles640"
model = YOLO("yolo26n.pt")
model.train(data=str(TILES / "data.yaml"),
            imgsz=640, epochs=100, patience=20, batch=-1, seed=42,
            device=0, cache="ram", workers=2,
            project=RUNS, name=RUN)
RUN = Path(model.trainer.save_dir).name
log_run(RUN)

res_tiles = run_eval(f"{RUNS}/{RUN}/weights/best.pt", VAL, imgsz=640, tile=640)
json.dump(res_tiles, open(f"{RUNS}/{RUN}/eval_val_sahi640.json", "w"), indent=2)
show(res_tiles)